# Compute signatures and pairwise whole-genome similarity using sourmash

## ____________________________________________________________________________________________

### Setup libraries, configuration, and directories

In [ ]:
## Import libraries

import sourmash
import subprocess
import tempfile
import pandas as pd
import numpy as np
import os
from multiprocessing import Pool
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import re
import time
import random
import logging
import shutil

In [ ]:
## Setup directories and configuration variables


# Configuration variables
K_VALUES = [16, 21, 31, 51, 61, 71]  
SCALED = 1000
SIGNATURE_TIMEOUT = 1800
SOURMASH_THREADS = 32

# Define working directory
BASE_DIR = Path("/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae")
os.chdir(BASE_DIR)


# Create main directories
ASSEMBLY_DIR = BASE_DIR / "Assemblies_SameSpecies"
TREE_DIR = BASE_DIR / "Sourmash_Hypocreaceae/SourmashTrees/SameSpecies"
SOURMASH_DIR = BASE_DIR / "Sourmash_Hypocreaceae/SourmashSignatures/SameSpecies"


# Create directories 
for directory in [ASSEMBLY_DIR, TREE_DIR, SOURMASH_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

### Define some necessary functions

In [ ]:
## Create a few functions for sourmash: computing signatures and ANI


# Compute signature for genome assembly
def compute_signature(fna_file, k):
    """Compute sourmash signature for a given k-mer size."""
    sig_path = SOURMASH_DIR / f"k{k}" / f"{fna_file.stem}_k{k}.sig"
    cmd = f"sourmash sketch dna -p k={k},scaled={SCALED} {fna_file} -o {sig_path}"
    try:
        subprocess.run(cmd, shell=True, check=True, capture_output=True, timeout=SIGNATURE_TIMEOUT)
        logging.info(f"Completed signature for {fna_file} with k={k}")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        logging.error(f"Failed to compute signature for {fna_file} with k={k}: {e}")


In [ ]:
# Main processing function for computing comprehensive pairwise ANI for all assemblies using all k-values and both ANI methods

def ComputeSignatures(accessions):
    """Compute signatures and all pairwise ANI comparisons for all accessions and k-values."""

    # Get all successfully extracted assemblies
    fna_files = list(ASSEMBLY_DIR.glob("*.fna")) + list(ASSEMBLY_DIR.glob("*.fasta"))
    logging.info(f"Found {len(fna_files)}/{len(accessions)} assemblies ready for processing")

    
    # Create k-value directories if needed
    for k in K_VALUES:
        (SOURMASH_DIR / f"k{k}").mkdir(exist_ok=True)
    
    # Compute signatures for all k-values
    logging.info("Computing signatures...")
    with ThreadPoolExecutor(max_workers=SOURMASH_THREADS) as executor:
        for fna_file in fna_files:
            for k in K_VALUES:
                executor.submit(compute_signature, fna_file, k)


In [ ]:
## Compute comprehensive ANI comparisons among all assemblies at different k's and ANI estimations

ComputeSignatures(os.listdir(ASSEMBLY_DIR))

In [ ]:
def compute_comprehensive_ani(k):
    """Compute ANI matrices for all available signatures at a given k-value."""
    k_dir = SOURMASH_DIR / f"k{k}"
    
    # Get all signature files for this k-value
    sig_files = list(k_dir.glob(f"*_k{k}.sig"))
    
    if not sig_files:
        logging.error(f"No signature files found for k={k}")
        return
    
    # Create signature list file
    sig_list = k_dir / "all_signatures.txt"
    with open(sig_list, "w") as f:
        for sig_file in sig_files:
            f.write(f"{sig_file}\n")
    
    # Compute Jaccard ANI matrix
    jaccard_out = k_dir / "comprehensive_jaccard.npy"
    jaccard_cmd = f"sourmash compare --from-file {sig_list} -k {k} --ani -o {jaccard_out}"
    subprocess.run(jaccard_cmd, shell=True, check=True)
    
    # Compute Max-Containment ANI matrix
    containment_out = k_dir / "comprehensive_maxcontainment.npy"
    containment_cmd = f"sourmash compare --from-file {sig_list} -k {k} --max-containment --ani -o {containment_out}"
    subprocess.run(containment_cmd, shell=True, check=True)
    
    # Save accession order for reference
    accessions = [sig_file.stem.replace(f"_k{k}", "") for sig_file in sig_files]
    with open(k_dir / "accession_order.txt", "w") as f:
        f.write("\n".join(accessions))
    
    logging.info(f"Completed ANI calculations for k={k} with {len(sig_files)} signatures")


In [ ]:
# Run for all k-values
for k in K_VALUES:
    compute_comprehensive_ani(k)


In [ ]:
os.listdir(ASSEMBLY_DIR)